In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# Dagshub/Mlflow initialization

In [4]:
!pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 76.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import dagshub
import mlflow
import mlflow.sklearn
import os

TOKEN = '773e6d0f45f4f8c8ed729b693f548555a4de31f8'

os.environ['DAGSHUB_USER_TOKEN'] = TOKEN
dagshub.auth.add_app_token(TOKEN)

dagshub.init(repo_owner='sbolk23', repo_name='IEEE-CIS-Fraud-Detection-Kaggle-Competition', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=3470309e-e866-4f51-b620-20f050d2242a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=55c4371358589e80f3fbe760fd04157442e1062e6c4317298fde0ed1ebff226f




KeyboardInterrupt: 

# Load Test

In [4]:
test_transaction_df = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity_df    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
test_df  = pd.merge(test_transaction_df, test_identity_df, on='TransactionID', how='left')

rename_dict = {f'id-{i:02d}': f'id_{i:02d}' for i in range(1, 39)}
test_df = test_df.rename(columns=rename_dict)
test_ids = test_df['TransactionID']

# Model Inference

In [5]:
import mlflow, sklearn

pipeline = mlflow.sklearn.load_model('models:/fraud_detection_final_xgboost/latest')
test_proba = pipeline.predict_proba(test_df)[:, 1]

submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud': test_proba,
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

/usr/local/lib/python3.12/dist-packages/mlflow/sklearn/__init__.py:550: UserWarning: [15:58:14] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  return cloudpickle.load(f)
/usr/local/lib/python3.12/dist-packages/mlflow/sklearn/__init__.py:550: UserWarning: [15:58:14] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  return cloudpickle.load(f)
/usr/local/lib/python3.12/dist-packages/mlflow/sklearn/__init__.py:550: UserWarning: [15:58:14] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  return cloudpickle.load(f)
